# 04b - cpm_tpm_like の QC（cell / gene 選択のみ）

`guess_preprocessing == "cpm_tpm_like"` の curated h5ad に対して stage1 QC を行う。

## 最重要方針

**QC では細胞と遺伝子を選択するだけ。発現量の値そのものは変更しない。**
stage1 QC 後の h5ad でも `.X` は CPM/TPM-like value のまま保持する。

```
保存する h5ad に対して以下は実行しない（禁止）:
  sc.pp.normalize_total(adata)
  sc.pp.log1p(adata)
  sc.pp.scale(adata)
```

## CPM/TPM-like での注意

CPM/TPM-like では `total_counts` は raw UMI 数ではない。
そのため stage1 QC では以下を除外条件に使わない:

```
total_counts
pct_counts_mt
pct_counts_ribo
```

ただし `sc.pp.calculate_qc_metrics` は実行し、出力された QC metrics は可視化する。

## 入出力

入力:
```
data/curated_h5ad/*.h5ad
results/preprocessing_state/preprocessing_state_summary.csv
```

出力:
```
data/qc_h5ad/cpm_tpm_like/<dataset_id>.stage1_flagged.h5ad
data/qc_h5ad/cpm_tpm_like/<dataset_id>.stage1_filtered.h5ad
```

`.X` は CPM/TPM-like のまま保存する。`sc.pp.filter_genes(adata, min_cells=3)` は遺伝子選択にのみ使う。


## セットアップ（パス・定数）

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sp
import matplotlib
matplotlib.use("Agg")  # 非対話バックエンド（保存優先）
import matplotlib.pyplot as plt

# v2 ルートを探す（config/dataset_manifest.yaml がある場所）
ROOT = Path.cwd()
while not (ROOT / "config" / "dataset_manifest.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

CURATED_DIR = ROOT / "data" / "curated_h5ad"
PREPROC_SUMMARY_CSV = ROOT / "results" / "preprocessing_state" / "preprocessing_state_summary.csv"

PREPROCESSING_STATE = "cpm_tpm_like"
QC_OUT_DIR = ROOT / "data" / "qc_h5ad" / PREPROCESSING_STATE
RESULTS_DIR = ROOT / "results" / "qc_original_scale_pipeline" / "04b_cpm_tpm_like"
PLOT_DIR = RESULTS_DIR / "plots"

for d in (QC_OUT_DIR, RESULTS_DIR, PLOT_DIR):
    d.mkdir(parents=True, exist_ok=True)

MAX_PLOT = 100_000          # 可視化用サンプリング上限（保存する h5ad は全細胞版）
RNG = np.random.default_rng(0)

print("project root :", ROOT)
print("curated dir  :", CURATED_DIR)
print("qc out dir   :", QC_OUT_DIR)
print("results dir  :", RESULTS_DIR)


## stage1 QC 条件（cell 選択は n_genes_by_counts のみ）

```
n_genes_by_counts >= 200
n_genes_by_counts <= 7000
gene filter min_cells = 3
```

`total_counts` / `pct_counts_mt` / `pct_counts_ribo` は除外条件に使わない（可視化のみ）。
`qc_reason_stage1`: low_genes / high_genes


In [ ]:
MIN_GENES = 200            # n_genes_by_counts >= 200
MAX_GENES = 7000           # n_genes_by_counts <= 7000
GENE_FILTER_MIN_CELLS = 3  # sc.pp.filter_genes(adata, min_cells=3)
# total_counts / pct_counts_mt / pct_counts_ribo は stage1 の除外条件には使わない


## gene symbol から mt / ribo フラグを作る

In [ ]:
def get_gene_symbols_upper(adata):
    if "gene_symbol_upper" in adata.var.columns:
        return adata.var["gene_symbol_upper"].astype(str).str.upper()
    if "gene_symbol" in adata.var.columns:
        return adata.var["gene_symbol"].astype(str).str.upper()
    return pd.Index(adata.var_names).astype(str).str.upper()


def add_gene_qc_flags(adata):
    gene_symbols = pd.Series(get_gene_symbols_upper(adata), index=adata.var_names)
    adata.var["mt"] = gene_symbols.str.startswith("MT-").values
    adata.var["ribo"] = gene_symbols.str.startswith(("RPS", "RPL", "MRPS", "MRPL")).values
    return adata


## 対象 preprocessing state の dataset を選ぶ

`preprocessing_state_summary.csv` の `guess_preprocessing` でフィルタする。

In [ ]:
summary_df = pd.read_csv(PREPROC_SUMMARY_CSV)
target = summary_df[summary_df["guess_preprocessing"].astype(str) == PREPROCESSING_STATE].copy()
print(f"{len(target)} datasets with guess_preprocessing == {PREPROCESSING_STATE!r}\n")
cols = [c for c in ["source_accession", "dataset_id", "guess_preprocessing", "confidence"] if c in target.columns]
print(target[cols].to_string(index=False) if len(target) else "(none)")


## QC 関数の定義

`sc.pp.calculate_qc_metrics` で QC 指標を計算し、boolean mask で `qc_pass_stage1` / `qc_reason_stage1` を作る。**`.X` の値は変更しない。**

In [ ]:
def compute_qc(adata):
    """calculate_qc_metrics を実行し obs に QC 指標を書き込む。.X は変更しない。"""
    adata = add_gene_qc_flags(adata)
    sc.pp.calculate_qc_metrics(
        adata,
        qc_vars=["mt", "ribo"],
        percent_top=[50, 100, 200, 500],
        log1p=True,
        inplace=True,
    )
    return adata


In [ ]:
def build_stage1_flags(adata):
    """qc_pass_stage1 / qc_reason_stage1 / qc_preprocessing_state を obs に付ける。

    CPM/TPM-like・log-normalized-like では total_counts はraw UMI数ではないため、
    stage1 では n_genes_by_counts だけで細胞を選択する。
    total_counts / pct_counts_mt / pct_counts_ribo は除外条件に使わない（可視化のみ）。
    .X の値は一切変更しない。
    """
    obs = adata.obs

    qc_pass_stage1 = (
        (obs["n_genes_by_counts"] >= MIN_GENES)
        & (obs["n_genes_by_counts"] <= MAX_GENES)
    )

    mask_low_genes = obs["n_genes_by_counts"] < MIN_GENES
    mask_high_genes = obs["n_genes_by_counts"] > MAX_GENES

    reason = (
        np.where(mask_low_genes, "low_genes,", "")
        + np.where(mask_high_genes, "high_genes,", "")
    )
    reason = pd.Series(reason, index=obs.index).str.rstrip(",")
    reason[reason == ""] = "pass"

    adata.obs["qc_pass_stage1"] = qc_pass_stage1.values
    adata.obs["qc_reason_stage1"] = reason.values
    adata.obs["qc_preprocessing_state"] = PREPROCESSING_STATE
    return adata


## 可視化関数（Scanpy）

In [ ]:
def sample_for_plot(adata):
    """巨大データでは可視化用に最大 MAX_PLOT 細胞をサンプリングする。"""
    if adata.n_obs > MAX_PLOT:
        idx = np.sort(RNG.choice(adata.n_obs, size=MAX_PLOT, replace=False))
        return adata[idx].copy()
    return adata


def save_qc_plots(adata, dataset_id):
    """Scanpy の関数で QC 指標を可視化して保存する（全細胞・QC 指標つきの flagged を入力）。"""
    adata_plot = sample_for_plot(adata)
    has_condition = (
        "Condition" in adata_plot.obs.columns
        and adata_plot.obs["Condition"].astype(str).nunique() > 1
    )

    violin_keys = [k for k in [
        "n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo",
        "pct_counts_in_top_50_genes", "pct_counts_in_top_100_genes",
        "pct_counts_in_top_200_genes", "pct_counts_in_top_500_genes",
    ] if k in adata_plot.obs.columns]
    try:
        sc.pl.violin(
            adata_plot,
            keys=violin_keys,
            groupby="Condition" if has_condition else None,
            rotation=90,
            show=False,
        )
        plt.savefig(PLOT_DIR / f"{dataset_id}_violin.png", dpi=100, bbox_inches="tight")
    except Exception as e:
        print(f"  [warn] violin failed: {e}")
    finally:
        plt.close("all")

    for x, y, color in [
        ("total_counts", "n_genes_by_counts", "pct_counts_mt"),
        ("total_counts", "pct_counts_mt", "Condition"),
        ("n_genes_by_counts", "pct_counts_mt", "Condition"),
    ]:
        use_color = None if (color == "Condition" and not has_condition) else color
        try:
            sc.pl.scatter(adata_plot, x=x, y=y, color=use_color, show=False)
            plt.savefig(PLOT_DIR / f"{dataset_id}_scatter_{x}_vs_{y}.png", dpi=100, bbox_inches="tight")
        except Exception as e:
            print(f"  [warn] scatter {x} vs {y} failed: {e}")
        finally:
            plt.close("all")

    try:
        sc.pl.highest_expr_genes(adata_plot, n_top=20, show=False)
        plt.savefig(PLOT_DIR / f"{dataset_id}_highest_expr_genes.png", dpi=100, bbox_inches="tight")
    except Exception as e:
        print(f"  [warn] highest_expr_genes failed: {e}")
    finally:
        plt.close("all")


## dataset ごとに QC を実行

1. curated h5ad を読む
2. `calculate_qc_metrics` で QC 指標を計算
3. `qc_pass_stage1` / `qc_reason_stage1` を作る
4. **stage1_flagged**（全細胞・QC 列追加・`.X` 不変）を保存
5. **stage1_filtered**（pass 細胞のみ・`filter_genes(min_cells=3)`・`.X` 不変）を保存

細胞と遺伝子を選択するだけで、発現量の値そのものは変更しない。

In [ ]:
PER_CELL_COLS = [
    "cell_uid", "source_accession", "dataset_id", "Condition", "qc_preprocessing_state",
    "total_counts", "log1p_total_counts", "n_genes_by_counts", "log1p_n_genes_by_counts",
    "pct_counts_mt", "pct_counts_ribo",
    "pct_counts_in_top_50_genes", "pct_counts_in_top_100_genes",
    "pct_counts_in_top_200_genes", "pct_counts_in_top_500_genes",
    "qc_pass_stage1", "qc_reason_stage1",
]

summary_rows = []
per_cell_frames = []

for _, row in target.iterrows():
    dataset_id = str(row["dataset_id"])
    source_accession = str(row.get("source_accession", dataset_id.split("_")[0]))
    h5ad_path = CURATED_DIR / f"{dataset_id}.h5ad"
    if not h5ad_path.exists():
        print(f"[skip] not found: {h5ad_path}")
        continue

    print(f"\n=== {source_accession}  {dataset_id} ===")
    adata = sc.read_h5ad(h5ad_path)
    n_cells_before = adata.n_obs
    n_genes_before = adata.n_vars

    # 1-3. QC 指標計算 + stage1 flag（.X 不変）
    adata = compute_qc(adata)
    adata = build_stage1_flags(adata)
    if "cell_uid" not in adata.obs.columns:
        adata.obs["cell_uid"] = adata.obs_names.astype(str)

    # 4. stage1 flagged 版を保存（全細胞を残し QC 列だけ追加。.X 不変）
    flagged_path = QC_OUT_DIR / f"{dataset_id}.stage1_flagged.h5ad"
    adata.write_h5ad(flagged_path)
    print(f"  flagged  -> {flagged_path.name}  ({adata.n_obs} cells x {adata.n_vars} genes)")

    # 可視化（flagged = 全細胞・QC 指標つき）
    save_qc_plots(adata, dataset_id)

    # 5-6. stage1 filtered 版を保存（qc_pass_stage1==True の細胞だけ。gene filter。.X 不変）
    adata_filt = adata[adata.obs["qc_pass_stage1"].values].copy()
    sc.pp.filter_genes(adata_filt, min_cells=GENE_FILTER_MIN_CELLS)
    n_genes_after = adata_filt.n_vars
    filtered_path = QC_OUT_DIR / f"{dataset_id}.stage1_filtered.h5ad"
    adata_filt.write_h5ad(filtered_path)
    print(f"  filtered -> {filtered_path.name}  ({adata_filt.n_obs} cells x {adata_filt.n_vars} genes)")

    obs = adata.obs
    n_pass = int(obs["qc_pass_stage1"].sum())
    summary_rows.append({
        "source_accession": source_accession,
        "dataset_id": dataset_id,
        "preprocessing_state": PREPROCESSING_STATE,
        "n_cells_before": n_cells_before,
        "n_cells_stage1_pass": n_pass,
        "fraction_stage1_pass": (n_pass / n_cells_before) if n_cells_before else float("nan"),
        "n_genes_before": n_genes_before,
        "n_genes_after_filter_genes_min_cells3": n_genes_after,
        "median_total_counts": float(obs["total_counts"].median()),
        "median_n_genes_by_counts": float(obs["n_genes_by_counts"].median()),
        "median_pct_counts_mt": float(obs["pct_counts_mt"].median()) if "pct_counts_mt" in obs.columns else float("nan"),
        "median_pct_counts_ribo": float(obs["pct_counts_ribo"].median()) if "pct_counts_ribo" in obs.columns else float("nan"),
    })

    pc = pd.DataFrame(index=obs.index)
    for col in PER_CELL_COLS:
        pc[col] = obs[col].values if col in obs.columns else np.nan
    per_cell_frames.append(pc)


## summary CSV と per-cell QC CSV を保存

In [ ]:
summary = pd.DataFrame(summary_rows)
summary_path = RESULTS_DIR / "cpm_tpm_like_qc_summary.csv"
summary.to_csv(summary_path, index=False)
print("saved:", summary_path)

if per_cell_frames:
    per_cell = pd.concat(per_cell_frames, axis=0, ignore_index=True)
else:
    per_cell = pd.DataFrame(columns=PER_CELL_COLS)
per_cell_path = RESULTS_DIR / "cpm_tpm_like_per_cell_qc.csv.gz"
per_cell.to_csv(per_cell_path, index=False, compression="gzip")
print("saved:", per_cell_path)
summary


## 次のステップ

`04d_merge_qc_original_scale.ipynb` で、各 preprocessing state の `.stage1_filtered.h5ad` を **正規化せず** に merge する（original-scale merge）。